In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    comment="The raw customer dataset, ingested from Unity Catalog volume."
)
def bronze_customers():
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ",")
        .load("/Volumes/workspace/db_test/customer_test/customers-100.csv")
    )

@dlt.table(
    comment="Cleaned schema and standardized customer fields."
)
@dlt.expect_or_drop("valid_subscription_date", "subscription_date IS NOT NULL")
def silver_customers():
    return (
        dlt.read("bronze_customers")
        .selectExpr(
            "`Customer Id` AS customer_id",
            "`First Name` AS first_name",
            "`Last Name` AS last_name",
            "Company AS company",
            "City AS city",
            "Country AS country",
            "`Phone 1` AS phone_1",
            "`Phone 2` AS phone_2",
            "Email AS email",
            "`Subscription Date` AS subscription_date",
            "Website AS website"
        )
        .withColumn("subscription_date", expr("CAST(subscription_date AS DATE)"))
    )

@dlt.table(
    comment="Gold table: yearly customer subscription counts."
)
def gold_customers_subscription_made_by_year():
    return (
        dlt.read("silver_customers")
        .groupBy(year(col("subscription_date")).alias("subscription_year"))
        .agg(count(lit(1)).alias("customer_count"))
        .orderBy(col("subscription_year"))
    )
